# 📚 Notebook 1: Introducción al Análisis de Sentimientos

## 🎯 Objetivos de este Notebook

En este notebook aprenderás:
1. ✅ ¿Qué es el **Análisis de Sentimientos**?
2. ✅ ¿Qué es **NLP** (Natural Language Processing)?
3. ✅ Cargar el **Dataset IMDB** de reviews de películas
4. ✅ Explorar las características del dataset
5. ✅ Ver ejemplos reales de reviews positivas y negativas

⏱️ **Tiempo estimado**: 15 minutos

---

## 💡 ¿Qué es el Análisis de Sentimientos?

El **Análisis de Sentimientos** es una técnica de NLP que permite identificar si un texto expresa una opinión **positiva**, **negativa** o **neutral**.

### Ejemplos:

| Review | Sentimiento |
|--------|-------------|
| *"Esta película es excelente, me encantó!"* | 🟢 **Positivo** |
| *"Terrible, pérdida de tiempo y dinero"* | 🔴 **Negativo** |
| *"Estuvo bien, nada especial"* | 🟡 **Neutral** |

### ¿Para qué sirve?

- 📱 **Redes sociales**: Analizar opiniones sobre productos
- 🎬 **Películas/Series**: Entender qué piensan los usuarios
- 🏨 **Hoteles/Restaurantes**: Monitorear satisfacción de clientes
- 📊 **Marcas**: Reputación online

---

## 🔧 Setup Inicial

Primero, vamos a importar las librerías necesarias y configurar el entorno.

In [4]:
# Importar librerías básicas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar visualizaciones
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# ============================================================================
# CONFIGURACIÓN DE RUTAS - Solución robusta para Jupyter
# ============================================================================
import os
import sys
from pathlib import Path

def find_project_root():
    """Encuentra la raíz del proyecto buscando config.py y src/"""
    current = Path.cwd()
    
    # Buscar hacia arriba hasta 5 niveles
    for _ in range(5):
        config_exists = (current / 'config.py').exists()
        src_exists = (current / 'src').exists()
        
        if config_exists and src_exists:
            return current
        
        # También verificar si estamos dentro de sentiment-analysis
        if current.name == 'sentiment-analysis' and src_exists:
            return current
            
        current = current.parent
    
    # Si no encuentra, asumir que es el directorio actual
    return Path.cwd()

# Encontrar y configurar la ruta del proyecto
project_root = find_project_root()
print(f"📁 Proyecto encontrado en: {project_root}")

# Agregar al path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Cambiar el working directory
os.chdir(str(project_root))

# Importar módulos
from src import data_loader

print("✅ Librerías importadas correctamente!")
print(f"📁 Directorio de trabajo: {os.getcwd()}")

📁 Proyecto encontrado en: /content


ModuleNotFoundError: No module named 'src'

## 📥 Cargar el Dataset IMDB

### ¿Qué es el Dataset IMDB?

- 📊 **50,000 reviews** de películas de [IMDB.com](https://www.imdb.com/)
- 🎬 Reviews reales escritas por usuarios
- ⚖️ **Balanceado**: 50% positivas, 50% negativas
- 📦 Viene incluido en **TensorFlow/Keras** (descarga automática)

### División del dataset:
- 🎓 **25,000 reviews** para entrenamiento
- 🧪 **25,000 reviews** para prueba

### Importante:
⚠️ La primera vez que ejecutes la siguiente celda, se descargará el dataset (~17 MB). Las siguientes veces usará el archivo ya descargado.

In [ ]:
from src import data_loader

print("📥 Cargando dataset IMDB...")
print("   (La primera vez descarga ~17 MB, puede tardar 1-2 minutos)\n")

# Cargar datos
(X_train, y_train), (X_test, y_test) = data_loader.load_imdb_data()

print("\n✅ ¡Dataset cargado exitosamente!")

## 🔍 ¿Qué contienen X_train e y_train?

- **X_train**: Las reviews (texto convertido a números)
- **y_train**: Las etiquetas (0 = negativo, 1 = positivo)

Veamos la forma de los datos:

In [ ]:
print("📊 INFORMACIÓN DEL DATASET")
print("=" * 60)

print(f"\n🎓 Datos de ENTRENAMIENTO:")
print(f"   • Reviews (X_train): {len(X_train):,} samples")
print(f"   • Etiquetas (y_train): {len(y_train):,} samples")

print(f"\n🧪 Datos de PRUEBA:")
print(f"   • Reviews (X_test): {len(X_test):,} samples")
print(f"   • Etiquetas (y_test): {len(y_test):,} samples")

print(f"\n📈 TOTAL: {len(X_train) + len(X_test):,} reviews")

## 🤔 ¿Por qué X_train contiene NÚMEROS y no TEXTO?

¡Excelente pregunta! Veamos un ejemplo:

In [ ]:
# Ver la primera review (está en números)
print("🔢 Primera review en formato NUMÉRICO:")
print(f"   {X_train[0][:50]}...")  # Primeros 50 números

print(f"\n📏 Longitud: {len(X_train[0])} palabras")
print(f"🏷️  Sentimiento: {'Positivo ✅' if y_train[0] == 1 else 'Negativo ❌'}")

### 🧠 Explicación:

**¿Por qué números y no texto?**

Las computadoras **no entienden texto**, solo números. El dataset IMDB ya viene **"tokenizado"** (convertido a números):

```
Texto original:  "The movie was excellent"
                  ↓ Tokenización
Números:         [1, 14, 22, 89]
```

Donde:
- `1` = "the"
- `14` = "movie" 
- `22` = "was"
- `89` = "excellent"

Cada palabra tiene un **índice único** en el vocabulario.

---

## 📖 Decodificar Reviews (Convertir Números → Texto)

Para **entender** las reviews, necesitamos convertirlas de vuelta a texto. Para esto usamos el **diccionario de palabras**:

In [ ]:
# Obtener diccionario palabra → índice
print("📖 Cargando diccionario de palabras...")
word_index = data_loader.get_word_index()

print(f"\n✅ Diccionario cargado: {len(word_index):,} palabras únicas")

# Ver algunos ejemplos del diccionario
print("\n📚 Ejemplos de palabras y sus índices:")
for word, idx in list(word_index.items())[:10]:
    print(f"   '{word}' → {idx}")

### Ahora decodifiquemos la primera review para verla en texto:

In [ ]:
# Decodificar primera review
decoded_review = data_loader.decode_review(X_train[0], word_index)

print("📝 REVIEW DECODIFICADA (texto legible):")
print("=" * 60)
print(decoded_review)
print("=" * 60)

print(f"\n🏷️  Sentimiento: {'Positivo ✅' if y_train[0] == 1 else 'Negativo ❌'}")

## 📊 Exploración del Dataset

Vamos a analizar las características del dataset usando la función `explore_imdb_data()`:

In [ ]:
# Explorar dataset completo
data_loader.explore_imdb_data(X_train, y_train, X_test, y_test)

## 📈 Visualización: Distribución de Sentimientos

Veamos si el dataset está balanceado:

In [ ]:
# Contar positivos y negativos
positivos_train = np.sum(y_train == 1)
negativos_train = np.sum(y_train == 0)

positivos_test = np.sum(y_test == 1)
negativos_test = np.sum(y_test == 0)

# Crear gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training
axes[0].bar(['Negativo', 'Positivo'], [negativos_train, positivos_train], 
            color=['#e74c3c', '#2ecc71'], alpha=0.7)
axes[0].set_title('Distribución - Training Set', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Número de Reviews')
axes[0].grid(axis='y', alpha=0.3)

for i, (label, count) in enumerate(zip(['Negativo', 'Positivo'], [negativos_train, positivos_train])):
    axes[0].text(i, count + 200, f'{count:,}\n({count/len(y_train)*100:.1f}%)', 
                 ha='center', fontsize=11, fontweight='bold')

# Test
axes[1].bar(['Negativo', 'Positivo'], [negativos_test, positivos_test], 
            color=['#e74c3c', '#2ecc71'], alpha=0.7)
axes[1].set_title('Distribución - Test Set', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Número de Reviews')
axes[1].grid(axis='y', alpha=0.3)

for i, (label, count) in enumerate(zip(['Negativo', 'Positivo'], [negativos_test, positivos_test])):
    axes[1].text(i, count + 200, f'{count:,}\n({count/len(y_test)*100:.1f}%)', 
                 ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ El dataset está perfectamente balanceado: 50% positivo, 50% negativo")

## 📏 Visualización: Distribución de Longitudes

¿Qué tan largas son las reviews?

In [ ]:
# Calcular longitudes
train_lengths = [len(review) for review in X_train]
test_lengths = [len(review) for review in X_test]

# Estadísticas
print("📏 ESTADÍSTICAS DE LONGITUD (en palabras)")
print("=" * 60)
print(f"\n🎓 Training Set:")
print(f"   • Mínima:  {min(train_lengths)} palabras")
print(f"   • Máxima:  {max(train_lengths)} palabras")
print(f"   • Media:   {np.mean(train_lengths):.1f} palabras")
print(f"   • Mediana: {np.median(train_lengths):.1f} palabras")

# Gráfico
plt.figure(figsize=(14, 5))

plt.hist(train_lengths, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
plt.axvline(np.mean(train_lengths), color='red', linestyle='--', linewidth=2, 
            label=f'Media: {np.mean(train_lengths):.1f}')
plt.axvline(np.median(train_lengths), color='orange', linestyle='--', linewidth=2, 
            label=f'Mediana: {np.median(train_lengths):.1f}')

plt.xlabel('Longitud (palabras)', fontsize=12)
plt.ylabel('Frecuencia', fontsize=12)
plt.title('Distribución de Longitud de Reviews', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 La mayoría de reviews tienen entre 100 y 300 palabras")

## 🎬 Ver Ejemplos de Reviews Reales

Vamos a decodificar algunas reviews para ver ejemplos reales de positivas y negativas:

In [ ]:
# Función helper para mostrar reviews
def mostrar_review(index, X, y, word_index, max_chars=500):
    """Muestra una review decodificada con formato"""
    decoded = data_loader.decode_review(X[index], word_index)
    sentiment = "Positivo ✅" if y[index] == 1 else "Negativo ❌"
    
    # Truncar si es muy larga
    if len(decoded) > max_chars:
        decoded = decoded[:max_chars] + "..."
    
    print(f"\n{'='*70}")
    print(f"📝 Review #{index} - {sentiment}")
    print(f"{'='*70}")
    print(decoded)
    print(f"{'='*70}")

# Buscar ejemplos positivos y negativos
indices_positivos = np.where(y_train == 1)[0][:3]  # Primeras 3 positivas
indices_negativos = np.where(y_train == 0)[0][:3]  # Primeras 3 negativas

print("\n🟢 EJEMPLOS DE REVIEWS POSITIVAS:")
for idx in indices_positivos:
    mostrar_review(idx, X_train, y_train, word_index)

print("\n\n🔴 EJEMPLOS DE REVIEWS NEGATIVAS:")
for idx in indices_negativos:
    mostrar_review(idx, X_train, y_train, word_index)

## 🎯 Ejercicio Interactivo

**Explora el dataset por tu cuenta:**

Cambia el número en `index` para ver diferentes reviews:

In [ ]:
# 👇 CAMBIA ESTE NÚMERO (0 a 24999):
index = 100

# Ver review
mostrar_review(index, X_train, y_train, word_index, max_chars=800)

## 📊 Resumen de lo Aprendido

En este notebook aprendiste:

✅ **Qué es el Análisis de Sentimientos**: Clasificar texto como positivo/negativo

✅ **Dataset IMDB**: 
- 50,000 reviews de películas
- Balanceado (50% pos, 50% neg)
- Dividido en train (25k) y test (25k)

✅ **Tokenización**: 
- Texto → Números (para que la computadora lo entienda)
- Cada palabra tiene un índice único

✅ **Decodificación**: 
- Números → Texto (para que nosotros lo entendamos)
- Usamos el diccionario de palabras

✅ **Características del dataset**:
- Longitud media: ~230 palabras
- Vocabulario: ~88,000 palabras únicas

---

## 🎓 Próximo Paso

En el **Notebook 2** aprenderás:
- 🧹 Cómo **preprocesar y limpiar** el texto
- 🔤 **Tokenización** detallada
- ✂️ **Stopwords** (remover palabras sin significado)
- 🌱 **Stemming y Lemmatization** (reducir palabras a raíz)

**¡Nos vemos en el siguiente notebook!** 🚀